In [1]:
import jax
import jax.numpy as jnp
import jax.scipy.stats as stats
import numpy as np
from tqdm import tqdm

import blackjax

observed = np.random.normal(10, 20, size=1_000)
def logdensity_fn(x):
    logpdf = stats.norm.logpdf(observed, x["loc"], x["scale"])
    return jnp.sum(logpdf)

# Build the kernel
step_size = 1e-3
inverse_mass_matrix = jnp.array([1., 1.])
nuts = blackjax.nuts(logdensity_fn, step_size, inverse_mass_matrix)

# Initialize the state
initial_position = {"loc": 1., "scale": 2.}
state = nuts.init(initial_position)

print("Starting NUTS sampling...")

# Iterate
rng_key = jax.random.key(0)
step = jax.jit(nuts.step)
_,_ = step(rng_key, state)  # Warmup JIT
with jax.profiler.trace("/tmp/jax_traces"):
    for i in tqdm(range(1_000)):
        nuts_key = jax.random.fold_in(rng_key, i)
        state, _ = step(nuts_key, state)
    
print(state.position)


Starting NUTS sampling...


2025-11-27 23:05:49.606857: E external/xla/xla/python/profiler/internal/python_hooks.cc:416] Can't import tensorflow.python.profiler.trace
 42%|████▏     | 423/1000 [01:47<02:26,  3.94it/s] 
2025-11-27 23:07:36.852675: E external/xla/xla/python/profiler/internal/python_hooks.cc:416] Can't import tensorflow.python.profiler.trace


KeyboardInterrupt: 